---
title: "Part 10: Combining, Reshaping & Time Series"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/13-combining-reshaping.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/13-combining-reshaping.ipynb)

Your manager emails on Monday morning: three spreadsheets, different schools, different formats -- "combine these and tell me which region is improving." The data you want lives across files, split by category, and needs reshaping before it says anything useful. Part 9 showed you how to derive new columns from one table. This notebook adds the operations you reach for when the data is already split across multiple tables or needs to change shape: `pd.concat`, `pd.merge`, `groupby`, `pivot_table`, and time-indexed data with `DatetimeIndex`.

Part 9 (`12-pandas-operations.ipynb`) built the column-level toolkit. What you build here -- multi-table joins and time-indexed aggregations -- is the direct input for Part 11 (`14-time-series.ipynb`), which goes deeper on resampling and forecasting.

> Callout markers used throughout this notebook are explained on the [book cover page](../../index.qmd#callout-guide).

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/university_analytics.csv")
df["average_marks"] = (df["midterm_score"] + df["final_score"] + df["project_score"]) / 3

# courses.csv is a thin reference table: one row per course: useful for joins
courses = pd.read_csv("data/courses.csv")
df.head()

::: {.callout-note collapse="true" icon=false}
## Learning Objectives

By the end of Part 10 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Stack DataFrames with `pd.concat`, by row and by column | Sec. 1 |
| 2 | Combine two tables on a shared key with `pd.merge`, and choose the right `how` | Sec. 2 |
| 2b | Predict row counts after a merge and diagnose unexpected fan-out | Sec. 2 |
| 3 | Split a dataset into groups and aggregate each one with `groupby` | Sec. 3 |
| 4 | Reshape a grouped result into a wide summary with `pivot_table` | Sec. 4 |
| 5 | Parse text into datetime values with `to_datetime` and extract date components | Sec. 5 |
| 6 | Build a `DatetimeIndex` and use it to slice a time series by date | Sec. 6 |
| 7 | Select rows with a partial date string or a date range | Sec. 7 |
| 8 | Change the time granularity of a series with `resample` | Sec. 8 |
:::


## 1. Concatenating DataFrames with `pd.concat`

`pd.concat` stacks DataFrames together. By default it stacks rows on top of each other (`axis=0`), the operation you need when the same columns show up in two separate files or two separate batches:

In [ ]:
male_students = df[df["gender"] == "M"]
female_students = df[df["gender"] == "F"]

recombined = pd.concat([male_students, female_students], axis=0)
print(f"male: {len(male_students)}, female: {len(female_students)}, combined: {len(recombined)}")

Passing `axis=1` stacks columns side by side instead, matching rows up by index. This is the shape you get when a calculation is done separately and needs joining back onto the original table:

In [ ]:
pass_fail = (df["average_marks"] >= 0.6).rename("passed")
with_pass_fail = pd.concat([df[["student_id", "average_marks"]], pass_fail], axis=1)
with_pass_fail.head()

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Concatenating by column without matching indexes first</span><br><br>
<code>pd.concat([df_a, df_b], axis=1)</code> lines rows up by index position, not by any shared key. If <code>df_a</code> and <code>df_b</code> were filtered, sorted, or reset differently beforehand, row 0 in one might not be row 0 in the other, and the result silently combines the wrong rows. <code>pd.merge</code> (Sec. 2) is the safer choice whenever there is an actual key column to join on.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Split and Recombine</span><br><br>

<b>Goal:</b> Split <code>df</code> into two DataFrames by <code>caste</code>: one for <code>"BC"</code> and one for everything else, using <code>.isin()</code> or a boolean mask. Concatenate them back with <code>pd.concat</code> and confirm the row count matches the original.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>bc_only = df[df["program"] == "BC"]
not_bc = df[df["program"] != "BC"]
recombined = pd.concat([bc_only, not_bc], axis=0)
len(recombined) == len(df)</pre>
</div>

In [ ]:
# TODO: split df by program == "Data Science", concat back, and confirm the row count matches
...

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: `pd.concat` preserves all columns and uses `axis` to choose the direction</span><br><br>
`pd.concat(objs, axis=0)` stacks DataFrames row by row -- every row from the second appended below the last row of the first. `axis=1` places them side by side as new columns. Passing `keys=[...]` adds a named outer index level so you can trace which original frame each row came from. When column sets differ, `concat` fills the missing values with `NaN` rather than raising an error.
</div>

## 2. Joining Data with `pd.merge`

`pd.merge` combines two tables on a shared key, the same idea as a SQL join. The `courses` table loaded above has one row per `course_id`; merging it onto `df` attaches metadata (credits, department, instructor) to every enrollment row:

In [ ]:
df_with_meta = pd.merge(df, courses, on="course_id", how="left")
df_with_meta[["student_id", "course_id", "course_x", "department", "credits"]].head()

The `how` parameter controls which rows survive when a key has no match on the other side. The diagram below maps each strategy to the rows it keeps.

The `how` argument decides which rows survive when a key on one side has no match on the other. `courses` here has a row for every `course_id` in `df`, so every `how` gives the same result; the difference only shows up once the two tables disagree:

<div style='background:#EAF7F0;border-left:5px solid #059669;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#059669;font-weight:bold'><i class="bi bi-journal-code"></i> Example: When <code>how</code> actually changes the result</span><br><br>
Merging <code>df</code> against only half of <code>courses</code> with <code>how="inner"</code> keeps only enrollments whose course is in that half. The same merge with <code>how="left"</code> keeps every student, with <code>region</code> set to <code>NaN</code> wherever the school was missing from the smaller table.
</div>

In [ ]:
half_courses = courses[courses["department"] == "Computing"]

inner_result = pd.merge(df, half_courses, on="course_id", how="inner")
left_result = pd.merge(df, half_courses, on="course_id", how="left")
missing = left_result["department"].isna().sum()
print(f"inner: {len(inner_result)} rows, left: {len(left_result)} rows, unmatched dept: {missing}")

The four join types differ only in which rows they keep, never in how rows are matched:

![The four pd.merge join types mapped by which rows survive. inner keeps matched rows only. left keeps all left rows with NaN where right has no match. right keeps all right rows. outer keeps every row from both sides with NaN filling the gaps.](figs/merge-join-types.svg){fig-alt="Four rows, one per join type: inner (green), left (blue), right (purple), outer (red). Each shows a Venn diagram, example result rows, a plain-English rule, and the how= label."}

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Forgetting that an unmatched key produces <code>NaN</code>, not a dropped row</span><br><br>
With <code>how="left"</code>, an enrollment whose <code>course_id</code> has no match in <code>courses</code> still gets a row, just with <code>NaN</code> in every column that came from <code>courses</code>. Code that assumes every row has a real <code>region</code> after a left merge will silently miscount or mis-group those rows later, usually in a <code>groupby</code> a few cells down.
</div>

### Duplicate keys: when merge produces more rows than you expect

Every example so far used a key where the right table had at most one match per left row (many-to-one). When the right table has more than one match, the result has more rows than either input, and this is correct but often surprising.

Concrete example: if a student is enrolled in three courses, a left merge of the enrollment table against a course details table produces three rows for that student, one per course. That is the right answer, but code that assumed row counts stay the same will silently operate on a bigger table.

In [ ]:
# Create a small example: two students, one appearing in two courses
students_small = pd.DataFrame(
    {
        "student_id": ["S0001", "S0001", "S0002"],
        "course_id": ["C01", "C02", "C01"],
        "score": [85, 72, 91],
    }
)

course_info = pd.DataFrame(
    {
        "course_id": ["C01", "C02"],
        "course_name": ["Statistics", "Algorithms"],
        "credits": [3, 4],
    }
)

merged_dup = pd.merge(students_small, course_info, on="course_id", how="left")
print(f"students_small rows: {len(students_small)}")
print(f"after merge rows   : {len(merged_dup)}")
merged_dup

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: A many-to-one merge preserves row count; a one-to-many merge multiplies it</span><br><br>
<b>Many-to-one (the common case):</b> each row on the left matches at most one row on the right. Row count stays the same.<br><br>
<b>One-to-many:</b> one row on the left matches several rows on the right. Each match becomes its own output row. A student enrolled in 3 courses becomes 3 rows after merging on a course-detail table.<br><br>
<b>Many-to-many:</b> both sides have duplicates on the key. Every left row pairs with every matching right row: 3 left rows matching 4 right rows produces 12 output rows. This is almost never what you want and is usually a sign of a design error in the data or the query.
</div>

In [ ]:
# Verify: check how many rows the real df gains after merging on course_id
before = len(df)
after = len(pd.merge(df, courses, on="course_id", how="left"))
print(f"Before merge: {before} rows")
print(f"After  merge: {after} rows")
print(f"Difference  : {after - before} extra rows (students with multiple course matches)")

# Diagnose: how many course_ids appear more than once in courses?
dupe_courses = courses[courses.duplicated("course_id", keep=False)]
print(f"\nDuplicate course_ids in courses.csv: {dupe_courses['course_id'].nunique()}")

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Summing a column after an unexpected fan-out</span><br><br>
If <code>df</code> has one row per student and <code>groupby("student_id")["score"].sum()</code> gives the right answer before a merge, but the merge doubles some rows, the same <code>sum()</code> after the merge silently counts those scores twice. The most reliable diagnostic is to compare <code>len(df)</code> before and after any merge. A row count increase without a planned one-to-many join is a bug, not a feature.
<pre style='background:#F4F5F6;padding:10px;border-radius:4px;font-size:0.9em'># Always verify after merging
before = len(df)
merged = pd.merge(df, other_table, on="key", how="left")
assert len(merged) == before, f"Unexpected fan-out: {before} -> {len(merged)} rows"</pre>
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Count Enrollments Per Department</span><br><br>

<b>Goal:</b> Merge <code>df</code> with <code>courses</code> using <code>how="left"</code>, then use <code>.value_counts()</code> on the resulting <code>department</code> column.egion</code> column to see how many student rows fall in each region.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>merged = pd.merge(df, courses, on="course_id", how="left")
merged["department"].value_counts()</pre>
</div>

In [ ]:
# TODO: merge df with courses (how="left"), then value_counts on department
...

## 3. Group By Operations

`groupby` splits a dataset into groups by a key, applies a function to each group independently, and combines the results back into one table, one row per group:

![groupby split-apply-combine pattern. A full DataFrame is split into three group sub-tables (CS, DS, IT), mean() is applied to each group independently, and the results are combined into a one-row-per-group Series.](figs/groupby-split-apply-combine.svg){fig-alt="Three-stage diagram. Left: full DataFrame with program and avg_marks columns, rows color-coded by group. Centre: three small group tables with mean() annotation. Right: result Series with one row per program."}

In [ ]:
df.groupby("program")["average_marks"].mean()

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>groupby</code> computes nothing until you aggregate</span><br><br>
<code>df.groupby("program")</code> on its own returns a <code>DataFrameGroupBy</code> object, a plan for splitting the data, not a result. Nothing is actually computed until a method like <code>.mean()</code>, <code>.sum()</code>, or <code>.agg()</code> is called on it, the same lazy-then-compute pattern you will see again in Part 11's Polars notebook.
</div>

`.agg()` works after `groupby` exactly as it did in Part 2, computing several statistics per group in one call:

In [ ]:
df.groupby("program")["average_marks"].agg(["mean", "std", "count"])

Grouping by more than one column splits into one group per combination, here gender within caste:

In [ ]:
df.groupby(["program", "gender"])["average_marks"].mean()

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: <code>observed=True</code> when grouping a <code>category</code> column</span><br><br>
Grouping a column with <code>category</code> dtype (Part 2, Sec. 2) by default includes every category the dtype knows about, even ones with zero rows in the current data, padding the result with empty groups. Passing <code>observed=True</code> to <code>groupby</code> keeps only the categories that actually appear.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Dropout Rate by Caste</span><br><br>

<b>Goal:</b> Group <code>df</code> by <code>caste</code> and compute, for each group, what fraction of students have <code>continue_drop == "drop"</code>. <code>(series == "drop").mean()</code> gives a fraction directly, no separate count and divide needed.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df.groupby("program")["continue_drop"].apply(lambda s: (s == "drop").mean())</pre>
</div>

In [ ]:
# TODO: group by program, compute the pass fraction per program
...

## 4. Pivoting Data

`pivot_table` is `groupby` plus a reshape, in one call: it groups by one column, splits further by another, and lays the second grouping out as columns instead of stacking it into more rows, much easier to scan as a summary:

In [ ]:
pd.pivot_table(df, index="program", columns="gender", values="average_marks", aggfunc="mean")

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>pivot_table</code> is <code>groupby</code> with a different output shape</span><br><br>
<code>df.groupby(["program", "gender"])["average_marks"].mean()</code> from Sec. 3 and the <code>pivot_table</code> call above compute the exact same numbers. <code>groupby</code> returns them stacked into a long Series with a two-level index; <code>pivot_table</code> lays the second key out across columns instead. Reach for <code>pivot_table</code> specifically when the result is meant to be read by a person, not processed further by code.
</div>

`aggfunc` accepts a list too, giving more than one statistic per cell:

In [ ]:
pd.pivot_table(df, index="program", columns="gender", values="average_marks", aggfunc=["mean", "count"])

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Region by Program Pivot</span><br><br>

<b>Goal:</b> Merge <code>df</code> with <code>courses</code> (Sec. 2), then build a pivot table with <code>region</code> as the index, <code>program</code> as the columns, and the mean <code>average_marks</code> in each cell.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>merged = pd.merge(df, courses, on="course_id", how="left")
pd.pivot_table(merged, index="region", columns="program", values="average_marks", aggfunc="mean")</pre>
</div>

In [ ]:
# TODO: merge with courses, then pivot region x program on mean final_score
...

## Capstone: Regional Performance Report

Combine every operation from this notebook into one report: a merge to bring in region, a groupby to summarize, and a pivot to lay the summary out for reading.

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Regional Performance Report</span><br><br>

<b>Goal:</b>
<ol>
<li>Merge <code>df</code> with <code>courses</code> on <code>course_id</code>, keeping every student (Sec. 2)</li>
<li>Group the merged table by <code>region</code> and compute the mean <code>average_marks</code> and the drop fraction, using the function from Activity 3 (Sec. 3)</li>
<li>Build a pivot table with <code>region</code> as rows and <code>caste</code> as columns, mean <code>average_marks</code> in each cell (Sec. 4)</li>
</ol>
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>merged = pd.merge(df, courses, on="course_id", how="left")

regional_summary = merged.groupby("region").agg(
    mean_marks=("average_marks", "mean"),
    drop_fraction=("continue_drop", lambda s: (s == "drop").mean()),
)

region_caste_pivot = pd.pivot_table(merged, index="region", columns="program", values="average_marks", aggfunc="mean")</pre>
</div>

In [ ]:
# TODO: build the regional performance report described above
...

## 5. Time Series Data

The student exam results dataset has no date column. The attendance dataset does: daily records for five schools across a full school term. Loading it alongside the exam data gives Part 10 two tables to work with and shows how the same pandas idioms scale to a new problem shape.

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: A time-series dataset has one row per point in time and uses a datetime column as its natural key</span><br><br>
Most DataFrames start with a date column stored as plain text (`object` dtype). Before you can filter by date, compute time deltas, or resample, you must convert it with `pd.to_datetime()`. Once converted, the column holds `Timestamp` values -- pandas' equivalent of `datetime.datetime` -- and arithmetic like `date + pd.Timedelta('7 days')` works directly.
</div>

In [ ]:
attendance = pd.read_csv("data/daily_attendance.csv")
attendance.dtypes

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 5a - Explore the Attendance Dataset</span><br><br>
<b>Goal:</b> Get a quick feel for the time-series dataset before converting dates.<br><br>
<b>TODO:</b> Print the shape of <code>attendance</code>, the range of <code>date</code> values (first and last), and how many unique schools are in <code>school_id</code>. No date conversion yet -- just exploration.
</div>

In [ ]:
# TODO: print shape, first and last date value, and number of unique schools
...

## 6. Date and Time Data Types

The `date` column above read in as plain text, `str` dtype, not a date pandas can do arithmetic on. `pd.to_datetime` converts it to pandas' dedicated datetime dtype, `datetime64`:

In [ ]:
attendance["date"] = pd.to_datetime(attendance["date"])
attendance.dtypes

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Pandas 3 infers the resolution it needs, not always nanoseconds</span><br><br>
Earlier pandas versions always stored datetimes as <code>datetime64[ns]</code>, nanosecond precision, whether the data needed it or not. Pandas 3's <code>to_datetime</code> infers a resolution from what is actually in the data: day-level strings like the ones here become <code>datetime64[s]</code> or coarser, not nanoseconds. <code>attendance["date"].dtype</code> shows whichever resolution was inferred for this column.
</div>

A single value out of a datetime column is a `Timestamp`, pandas' equivalent of Python's `datetime.datetime`, with the same year, month, day, and weekday attributes:

In [ ]:
first_day = attendance["date"].iloc[0]
print(type(first_day))
print(f"year={first_day.year}, month={first_day.month}, day_name={first_day.day_name()}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 5 - Parse and Inspect</span><br><br>

<b>Goal:</b> Convert a list of three date strings, <code>["2025-01-06", "2025-02-14", "2025-03-28"]</code>, to <code>Timestamp</code> values with <code>pd.to_datetime</code>, then print the day name of each one.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>dates = pd.to_datetime(["2025-01-06", "2025-02-14", "2025-03-28"])
for d in dates:
    print(d.day_name())</pre>
</div>

In [ ]:
# TODO: convert the three date strings and print each day name
...

## 7. The `DatetimeIndex`

Setting a datetime column as the index turns it into a `DatetimeIndex`, which unlocks date-based slicing instead of only position-based or exact-label lookups. Each school's rows are pulled out first, since a `DatetimeIndex` only makes sense for one time series at a time:

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Setting a datetime column as the index creates a `DatetimeIndex` that unlocks date-based slicing</span><br><br>
Once you call `.set_index('date')` on a datetime column, the result has a `DatetimeIndex`. You can then use partial date strings in `.loc` -- `school.loc['2025-02']` selects every row in February without spelling out start and end times. Without a `DatetimeIndex`, `.loc` requires an exact label match; with one, it understands ranges.
</div>

In [ ]:
school_300 = attendance[attendance["school_id"] == 300].set_index("date")
school_300.index

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Setting the index before filtering to one entity</span><br><br>
<code>attendance.set_index("date")</code> on the full table produces a <code>DatetimeIndex</code> with the same date repeated once per school, since every school has a row for every day. Slicing that index by date then returns rows from every school mixed together for that date, not a clean single time series. Filter to one entity first, exactly as <code>school_300</code> does above, then set the index.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 6 - Build Another School's Series</span><br><br>

<b>Goal:</b> Filter <code>attendance</code> to <code>school_id == 302</code>, set <code>date</code> as the index, and confirm the result's index is a <code>DatetimeIndex</code> with <code>isinstance(result.index, pd.DatetimeIndex)</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>school_302 = attendance[attendance["school_id"] == 302].set_index("date")
isinstance(school_302.index, pd.DatetimeIndex)</pre>
</div>

In [ ]:
# TODO: filter to school_id 302, set date as index, confirm DatetimeIndex
...

## 8. Selecting Data from a Time Series

A `DatetimeIndex` accepts a partial date string in `.loc`, matching every row that falls inside it. `"2025-02"` selects the whole month without spelling out the first and last day:

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: A `DatetimeIndex` accepts partial date strings in `.loc`, matching every row in the period</span><br><br>
`school.loc['2025-02']` returns all rows in February 2025. `school.loc['2025-02-01':'2025-02-07']` returns a week-long slice, inclusive of both endpoints. This partial-string matching works at year, month, or day granularity -- you never need to construct explicit `datetime` objects for basic slicing.
</div>

In [ ]:
school_300.loc["2025-02"].head()

A slice with two partial dates selects everything between them, inclusive of both ends:

In [ ]:
school_300.loc["2025-02-01":"2025-02-07"]

<div style='background:#EAF7F0;border-left:5px solid #059669;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#059669;font-weight:bold'><i class="bi bi-journal-code"></i> Example: Comparing the size of two date ranges</span><br><br>
<code>len(school_300.loc["2025-01"])</code> against <code>len(school_300.loc["2025-02"])</code> confirms the row count for each month matches its number of business days, the same `bdate_range` weekday-only pattern used to build this dataset in the first place.
</div>

In [ ]:
print(f"January business days  : {len(school_300.loc['2025-01'])}")
print(f"February business days : {len(school_300.loc['2025-02'])}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 7 - Filter the Last Two Weeks of Term</span><br><br>

<b>Goal:</b> Select every row in <code>school_300</code> from <code>"2025-03-15"</code> to <code>"2025-03-28"</code> inclusive, and print the mean <code>attendance_rate</code> over that range.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>last_two_weeks = school_300.loc["2025-03-15":"2025-03-28"]
last_two_weeks["attendance_rate"].mean()</pre>
</div>

In [ ]:
# TODO: select 2025-03-15 through 2025-03-28 and print the mean attendance_rate
...

## 9. The Power of Pandas: `resample`

`resample` changes the time granularity of a series: daily data summarized into weekly or monthly figures, the same split-apply-combine idea from Part 3's `groupby`, except the groups are time intervals instead of category values:

In [ ]:
weekly_attendance = school_300["attendance_rate"].resample("W").mean()
weekly_attendance.head()

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>resample</code> groups by time interval, <code>groupby</code> groups by value</span><br><br>
<code>df.groupby("program")</code> (Part 3) splits rows by whatever value is already in the <code>caste</code> column. <code>series.resample("W")</code> splits rows by which week their <code>DatetimeIndex</code> label falls into, intervals that did not exist as a column at all until <code>resample</code> created them. Both still end with an aggregation like <code>.mean()</code> to combine each group into one number.
</div>

Monthly resampling on the same series needs only a different frequency string. In pandas 3 the old `"M"` alias was removed; the replacement is `"ME"` (month-end), which anchors each bucket to the last calendar day of the month:

In [ ]:
monthly_attendance = school_300["attendance_rate"].resample("ME").mean()
monthly_attendance

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Resampling the whole DataFrame keeps every entity separate, with care</span><br><br>
<code>attendance.set_index("date").groupby("school_id")["attendance_rate"].resample("W").mean()</code> resamples each school's series independently in one call, instead of looping over schools and resampling each one by hand. <code>groupby</code> before <code>resample</code> is what keeps the schools from being averaged together.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 8 - Monthly Comparison Across Schools</span><br><br>

<b>Goal:</b> Set <code>date</code> as the index on the full <code>attendance</code> table, group by <code>school_id</code>, and resample to monthly means in one chained call. Use <code>"ME"</code> (month-end): the pandas 3 replacement for the old <code>"M"</code> alias.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>attendance.set_index("date").groupby("school_id")["attendance_rate"].resample("ME").mean()</pre>
</div>

In [ ]:
# TODO: set date as index, group by school_id, resample monthly, take the mean
...

## Capstone: Term-End Attendance Report

Combine every operation from this notebook: parsing dates, building a per-school time series, slicing by date, and resampling, into one short report comparing the start and end of term.

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Term-End Attendance Report</span><br><br>

<b>Goal:</b>
<ol>
<li>Set <code>date</code> as the index on the full <code>attendance</code> table (Sec. 2)</li>
<li>Group by <code>school_id</code> and resample to weekly means (Sec. 4)</li>
<li>From the result, select the first week of January and the last week of March for every school, using a partial date string (Sec. 3)</li>
<li>Report which school had the largest drop in attendance between those two weeks</li>
</ol>
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>weekly = attendance.set_index("date").groupby("school_id")["attendance_rate"].resample("W").mean()

first_week = weekly.loc[:, "2025-01-06":"2025-01-12"]
last_week = weekly.loc[:, "2025-03-24":"2025-03-28"]
drop_per_school = first_week.groupby("school_id").mean() - last_week.groupby("school_id").mean()
drop_per_school.sort_values(ascending=False)</pre>
</div>

In [ ]:
# TODO: build the term-end attendance report described above
...

## Further Reading

| Resource | Why it matters |
|---|---|
| McKinney, W. (2022). *Python for Data Analysis*, 3rd ed. O'Reilly. | Chapter 8 (data wrangling) and Chapter 10 (aggregation) are the canonical references for `concat`, `merge`, and `groupby` |
| [pandas documentation: Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html) | Full API reference with worked examples for every `how=` variant and join edge-case |
| Wickham, H. (2014). [Tidy data](https://doi.org/10.18637/jss.v059.i10). *Journal of Statistical Software* 59(10). | Free PDF: defines when data is "tidy" and explains `.melt()` / `.pivot()` as transformations toward tidy form |
| [pandas documentation: Reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html) | `.pivot_table()`, `.stack()`, `.unstack()`, and `.crosstab()` in one place |


## Summary

| Concept | Key rule |
|---|---|
| `pd.concat(axis=0)` | Stack rows; use when the same columns appear in separate batches |
| `pd.concat(axis=1)` | Stack columns by index position, not by key; prefer `merge` when there is a real key |
| `pd.merge(..., on=key, how=...)` | Combine two tables on a shared key, the same idea as a SQL join |
| `how="inner"` | Keep only rows with a match on both sides |
| `how="left"` | Keep every row from the left table; unmatched rows get `NaN` |
| Many-to-one merge | Row count stays the same; each left row matches at most one right row |
| One-to-many merge | Row count grows; each left row can match multiple right rows |
| Fan-out check | Always compare `len(before)` vs `len(after)` when a row count increase is not expected |
| `groupby(...)` | Returns a plan, not a result; nothing computes until you aggregate |
| `.agg([...])` after `groupby` | Several statistics per group, in one call |
| `pivot_table` | `groupby` with the second key laid out as columns instead of stacked as rows |
| `pd.to_datetime` | Parses text into pandas' `datetime64` dtype; pandas 3 infers the resolution |
| `Timestamp` | A single datetime value, with `.year`, `.month`, `.day_name()`, and similar attributes |
| `DatetimeIndex` | Set a datetime column as the index to unlock date-based slicing |
| Filter before indexing | Set a `DatetimeIndex` on one entity's rows, not a table mixing several entities |
| `.loc["2025-02"]` | A partial date string selects every row inside that period |
| `.loc[start:end]` | A date range slice is inclusive of both ends |
| `.resample(freq)` | Groups rows by time interval instead of by value, then needs an aggregation like `.mean()` |
| `groupby(...).resample(...)` | Resample each entity's series independently in one chained call |

**Next:** `15-polars.ipynb`, covering Polars' DataFrame, expression API, and lazy evaluation.